 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison 
Name           : Ross Moroney                <br>
Student Number : L00196752       <br>
Due Date       : 14th May                <br>
Assignment     : CA2             <br>
Module         : AI for Vision and NLP    <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level
An image of your working pipeline at high level can be inserted here



# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [1]:
# pip installs
!pip install scikit-learn
!pip install pandas

## Imports

In [2]:
import pandas as pd
import sys
print(sys.executable)
import pymupdf
import os
import site
print(site.getusersitepackages())
import cv2
import matplotlib.pyplot as plt
# allows inline plotting
%matplotlib inline

import pytesseract
from pytesseract import Output
pytesseract.pytesseract.tesseract_cmd = r"C:\Tesseract\tesseract.exe"

# regular expression
import re



# To allow for manual file upload selection
import tkinter as tk
from tkinter import filedialog


# For text preporcessing
import spacy
# for text analysis
from spacy.matcher import Matcher
from spacy import displacy


# For feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer



C:\Users\rossm\anaconda3\envs\ocr_env\python.exe
C:\Users\rossm\AppData\Roaming\Python\Python310\site-packages


# Support Functions

In [3]:
print("cv2", cv2.__version__)
print("pytesseract", pytesseract.__version__)


# Setting global params for matplotlib
plt.rcParams['figure.figsize'] = (20, 20)



cv2 4.13.0
pytesseract 0.3.13


# NLP

## Tesserect

In [4]:

nlp = spacy.load("en_core_web_sm")


# FILE SELECTION
root = tk.Tk()
root.withdraw() 
file_path = filedialog.askopenfilename(filetypes=[("PDF/Image files", "*.pdf *.jpg *.png *.jpeg")])


def extract_text_from_file(file_path):
    full_text = ""
    sample_img = None
    custom_config = r'--oem 1 --psm 6'
    
    
    # Process PDFs page by page
    if file_path.lower().endswith(".pdf"):
        doc = pymupdf.open(file_path)
        print(f"Processing {len(doc)} pages...")

        for page_num in range(len(doc)):
            pdf_page = doc[page_num]

            # Render page as high-resolution image for OCR accuracy
            page_image = pdf_page.get_pixmap(dpi=300)
            page_image.save("temp_ocr_image.jpg")
            img = cv2.imread("temp_ocr_image.jpg")
            
            # Keep the first page image as sample for the output
            if page_num == 0:
                sample_img = img
            
            page_text = pytesseract.image_to_string(img, config=custom_config)
            full_text += page_text + "\n"
            
    else:
        # If it's just a single image file
        sample_img = cv2.imread(file_path)
        full_text = pytesseract.image_to_string(sample_img, config=custom_config)

    # OUTPUT 1: Print to screen
    print("\n--- RAW OCR TEXT (First 500 chars) ---")
    print(full_text[:500] + "...") 
    
    return full_text, sample_img

# 3. EXECUTION
if file_path:
    print(f"You selected: {file_path}")
    
    # Process the file
    text_output, image_output = extract_text_from_file(file_path)

    # Create the spaCy object
    doc_object = nlp(text_output) 
    print("\nDoc object created successfully for NLP analysis.")

else:
    print("No file was selected.")

You selected: C:/Users/rossm/Downloads/ATU College/NLP/CA 2 - 12th May/sample documents/Taj_Mahal.jpeg

--- RAW OCR TEXT (First 500 chars) ---
3 Po Da

= & ~~ =

I & a ‘ mm

[ 2. eee 2

' . el oleae eee

; Cala: =~ E Ee

eee 7 i Ay le ae 4 a
oo J, an ee Oe
a == ree) od CR Lees dd ees ee
ge eS — . <a Se =

= rs 3 ee, tC ¥ SOO — EE ae = =
...

Doc object created successfully for NLP analysis.


In [5]:

nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    doc = nlp(text)

    # Preprocessing (From Lab 2b and 3b)
    # Tokenization, Lemmatization, and Stop Word removal
    cleaned_tokens = [token.lemma_.lower() for token in doc 
                      if not token.is_stop and not token.is_punct]


    # OUTPUT 2: See the cleaned tokens
    print("\n--- CLEANED TOKENS (No Stop Words + Lemmatized) ---")
    print(cleaned_tokens[:20]) 
    
    # OUTPUT 3: Named Entity Recognition (Visual output from Lab 2a)
    print("\n--- VISUALIZING ENTITIES ---")
    displacy.render(doc, style="ent", jupyter=True)
    
    return doc


    
# --- TRIGGER THE OUTPUT ---
# text_output' was created in the previous step
if 'text_output' in locals():
    doc_object = preprocess_text(text_output)
else:
    print("Error: 'text_output' not found. Please run the OCR cell first.")


--- CLEANED TOKENS (No Stop Words + Lemmatized) ---
['3', 'po', 'da', '\n\n', '=', '~~', '=', '\n\n', 'mm', '\n\n', '2', 'eee', '2', '\n\n', 'el', 'oleae', 'eee', '\n\n', 'cala', '=']

--- VISUALIZING ENTITIES ---


In [6]:
# Create the variable 'cleaned_lemmas' so the next cell can see it
cleaned_lemmas = [
    token.lemma_.lower() 
    for token in doc_object 
    if not token.is_stop and not token.is_punct and token.text.strip()
]

print("Successfully created cleaned_lemmas!")

Successfully created cleaned_lemmas!


In [7]:
# Convert the list of lemmas back into a single string for analysis
processed_text_for_tfidf = " ".join(cleaned_lemmas)

# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform([processed_text_for_tfidf])

# Get features and their scores
feature_names = vectorizer.get_feature_names_out()
scores = tfidf_matrix.toarray().flatten()

# Create a DataFrame for easy interpretation (Rubric requirement)
tfidf_df = pd.DataFrame({'keyword': feature_names, 'score': scores})
tfidf_df = tfidf_df.sort_values(by='score', ascending=False).head(10)

print("--- TOP 10 KEYWORDS BY TF-IDF SCORE ---")
display(tfidf_df)

--- TOP 10 KEYWORDS BY TF-IDF SCORE ---


,keyword,score
6,ee,0.717137
7,eee,0.358569
0,ae,0.239046
1,ay,0.119523
3,cr,0.119523
2,cala,0.119523
5,dd,0.119523
4,da,0.119523
8,el,0.119523
9,es,0.119523


In [8]:

def display_features_and_ner(text, spacy_doc):
    
    # 1. Named Entity Recognition (NER)
    entities = [(ent.text, ent.label_) for ent in spacy_doc.ents]
    
    print("\n--- NAMED ENTITY RECOGNITION (NER) ---")
    if entities:
        for ent_text, ent_label in entities:
            print(f"Found: {ent_text:20} | Category: {ent_label}")
    else:
        print("No entities found.")


    
    # 2. TF-IDF Feature Extraction
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text])
    feature_names = vectorizer.get_feature_names_out()
    
    print("\n--- TEXT FEATURE EXTRACTION (TF-IDF) ---")
    print(f"Top 20 Vocabulary Features: {list(feature_names)[:20]}")
    
    return entities, feature_names

# Extract named entities and generate TF-IDF features from the processed text
named_entities, tfidf_features = display_features_and_ner(text_output, doc_object)


--- NAMED ENTITY RECOGNITION (NER) ---
Found: 3                    | Category: CARDINAL
Found: 2                    | Category: CARDINAL
Found: 2                    | Category: CARDINAL
Found: el oleae             | Category: ORG
Found: Cala                 | Category: PERSON
Found: 7                    | Category: CARDINAL
Found: 4                    | Category: CARDINAL
Found: CR Lees              | Category: ORG
Found: a Se =               | Category: PERSON
Found: 3 ee, tC ¥           | Category: TIME
Found: SOO                  | Category: ORG

--- TEXT FEATURE EXTRACTION (TF-IDF) ---
Top 20 Vocabulary Features: ['ae', 'an', 'ay', 'cala', 'cr', 'da', 'dd', 'ee', 'eee', 'ees', 'el', 'es', 'ge', 'le', 'lees', 'mm', 'od', 'oe', 'oleae', 'oo']


In [9]:
def extract_document_references(doc):
    reference_matcher = Matcher(nlp.vocab)
    
    # Match references such as "Figure 1", "Fig. 2", or "Table 3",
    # allowing optional punctuation and up to three intermediate tokens
    reference_pattern = [
        {"LOWER": {"IN": ["figure", "fig", "table"]}},
        {"IS_PUNCT": True, "OP": "?"},
        {"OP": "{,3}"}, 
        {"IS_DIGIT": True}
    ]
    
    reference_matcher.add("RefMatcher", [reference_pattern])
    matches = reference_matcher(doc)
    
    # Store the matches in 'matched_references'
    matched_references = [doc[start:end].text for match_id, start, end in matches]
    
    print("\n--- REFERENCE ANALYSIS ---")
    print(f"Total references found (including duplicates): {len(matched_references)}")

    # Remove duplicates for summary reporting
    unique_references = sorted(list(set(matched_references)))
    
    print(f"The document contains {len(unique_references)} unique figures/tables:")
    print(unique_references)
    
    return matched_references

# Extract figure and table references from the processed document
document_references = extract_document_references(doc_object)


--- REFERENCE ANALYSIS ---
Total references found (including duplicates): 0
The document contains 0 unique figures/tables:
[]


In [10]:
# Structured summary
summary_data = {
    "Metric": ["Raw Text Length", "Total Tokens", "Unique Cleaned Lemmas", "Named Entities Found"],
    "Value": [len(text_output), len(doc_object), len(set(cleaned_lemmas)), len(doc_object.ents)]
}
summary_df = pd.DataFrame(summary_data)

print("--- ANALYSIS SUMMARY ---")
display(summary_df)

--- ANALYSIS SUMMARY ---


,Metric,Value
0,Raw Text Length,196
1,Total Tokens,86
2,Unique Cleaned Lemmas,35
3,Named Entities Found,11


# Vision

## Sub Heading 1

In [11]:
# code here...

# Multi-modal

## Sub Heading 1

In [12]:
# code here

# Final Output

In [13]:
# code